In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("powerplant_data.csv")
# df.head()
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB


In [3]:
X = df.drop("PE",axis=1)
y= df["PE"]

x_train ,x_test , y_train,y_test = train_test_split(X,y,test_size= 0.2,random_state =42)
x_train

,AT,V,AP,RH
5487,25.24,63.47,1011.30,66.21
3522,26.09,70.40,1007.41,85.37
6916,26.63,73.68,1015.15,85.13
7544,32.06,71.85,1007.90,56.44
7600,28.70,71.64,1007.11,69.85
...,...,...,...,...
5734,26.25,61.02,1011.47,71.22
5191,29.17,64.79,1016.43,61.05
5390,18.00,43.70,1015.40,61.28
860,26.73,68.84,1010.75,66.83


In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled =scaler.transform(x_test)

In [5]:
import torch
import torch.nn as nn

x_train_tensor = torch.tensor(x_train_scaled , dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values , dtype= torch.float32).view(-1,1)

x_test_tensor = torch.tensor(x_test_scaled , dtype= torch.float32)
y_test_tensor = torch.tensor(y_test.values , dtype = torch.float32).view(-1,1)



In [6]:
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(x_train_tensor , y_train_tensor)
test_dataset = TensorDataset(x_test_tensor , y_test_tensor)

train_loader = DataLoader(train_dataset,batch_size = 32,shuffle =True)
test_loader= DataLoader(test_dataset , batch_size = 32)

In [7]:
# build the architecture of model 
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(x_train.shape[1],6),
            nn.ReLU(),

            # 2nd 
            nn.Linear(6,6),
            nn.ReLU(),

            # o/p layer
            nn.Linear(6,1)
        )

    

    def forward(self,x):
        return self.model(x)

In [10]:
import torch.optim as optim
model = ANN()

# loss function
crietrion = nn.MSELoss()
# optimizer
optimizer = optim.Adam(model.parameters(),lr=0.001) 

In [13]:
# training the model
epochs =100
best_val_loss = float("inf")
for epoch in range(epochs):
    model.train()
    running_time_loss = 0.0  #for 1 epoch
    

    for xb,yb in train_loader:
        optimizer.zero_grad()
        output = model(xb)
        loss = crietrion(output,yb)  #compute loss
        loss.backward() #backward propogation ...gradient compute
        optimizer.step() #param update
        
        running_time_loss += loss.item() # loss tensor -> py float
    epoch_training_loss = running_time_loss/len(train_loader)

    running_val_loss = 0.0
    # validation loss
    model.eval()
    with torch.no_grad():
        for xb,yb in test_loader:
            output = model(xb)
            loss = crietrion(output,yb)
            running_val_loss +=loss
    epoch_val_loss = running_val_loss / len(test_loader)
        
    print(f"Training losses per epoch {epoch} =  {epoch_training_loss} ,  Testing loss {epoch_val_loss}")

    if epoch_val_loss<best_val_loss :
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(),"best_model.pt")
        


        
        
        

Training losses per epoch 0 =  20.77963917652766 ,  Testing loss 19.102294921875
Training losses per epoch 1 =  20.84550817410151 ,  Testing loss 18.85283660888672
Training losses per epoch 2 =  20.761655700206756 ,  Testing loss 18.819849014282227
Training losses per epoch 3 =  20.739083846410114 ,  Testing loss 19.424551010131836
Training losses per epoch 4 =  20.641781489054363 ,  Testing loss 18.9281005859375
Training losses per epoch 5 =  20.869685677687325 ,  Testing loss 19.280406951904297
Training losses per epoch 6 =  20.733360115687052 ,  Testing loss 21.3154239654541
Training losses per epoch 7 =  20.790826312700908 ,  Testing loss 18.8802547454834
Training losses per epoch 8 =  20.753462521235146 ,  Testing loss 18.92523765563965
Training losses per epoch 9 =  20.68278325398763 ,  Testing loss 18.902164459228516
Training losses per epoch 10 =  20.73796138763428 ,  Testing loss 19.6583251953125
Training losses per epoch 11 =  20.757668654123943 ,  Testing loss 18.98648071289

In [14]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [16]:
# evaluation
model.eval()
with torch.no_grad():
    train_pred = model(x_train_tensor)
    test_pred = model(x_test_tensor)

    train_loss = crietrion(train_pred,y_train_tensor)
    test_loss = crietrion(test_pred,y_test_tensor)
    print("train loss : ",train_loss)
    print("test loss : ",test_loss)

train loss :  tensor(20.5169)
test loss :  tensor(18.7750)


In [18]:
from sklearn.metrics import r2_score
print("Training r2_score : ",r2_score(y_train,train_pred))
print("Testing r2_score : ",r2_score(y_test,test_pred))

Training r2_score :  0.9298621346830874
Testing r2_score :  0.9343860878416725
